# 01 — Thu thập và kiểm tra dữ liệu BigMart

Notebook này ghi nhận nguồn dữ liệu đã tải và xác minh bản CSV cục bộ trước khi phân tích. Tải `bigmart.csv` từ [Kaggle BigMart Sales Dataset](https://www.kaggle.com/datasets/yasserh/bigmartsalesdataset), đặt nguyên bản tại `data/tu_data/raw/bigmart.csv`, và không chỉnh sửa/ghi đè tệp gốc. Metadata và SHA-256 nằm trong `source.json` cùng thư mục.

> Bản dữ liệu trong repository được lưu từ ZIP Kaggle. Hãy xem điều khoản của nguồn dữ liệu trước khi phân phối lại; notebook không tự tải dữ liệu hoặc gửi dữ liệu ra bên ngoài.

In [1]:
from pathlib import Path
import hashlib
import json

ROOT = Path.cwd().resolve()
while not (ROOT / "data/tu_data/raw/bigmart.csv").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Mở notebook trong repository có data/tu_data/raw/bigmart.csv.")
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data/tu_data/raw"
csv_path = RAW_DIR / "bigmart.csv"
metadata_path = RAW_DIR / "source.json"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy metadata nguồn: {metadata_path}")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
if sha256 != metadata.get("sha256"):
    raise ValueError("SHA-256 không khớp source.json; không tiếp tục với dữ liệu đã thay đổi.")

print(f"Dataset: {metadata['dataset']}")
print(f"Nguồn: {metadata['source']}")
print(f"Tệp: {csv_path.relative_to(ROOT)}")
print(f"Kích thước: {csv_path.stat().st_size:,} bytes")
print(f"SHA-256: {sha256}")

Dataset: Bigmart Sales Dataset
Nguồn: https://www.kaggle.com/datasets/yasserh/bigmartsalesdataset
Tệp: data\tu_data\raw\bigmart.csv
Kích thước: 869,537 bytes
SHA-256: 6f1b47d794ff393663517a075c3648baf413d0f1d687878b21c806d293ec2e41


In [2]:
import pandas as pd

expected_columns = [
    "Item_Identifier", "Item_Weight", "Item_Fat_Content", "Item_Visibility",
    "Item_Type", "Item_MRP", "Outlet_Identifier", "Outlet_Establishment_Year",
    "Outlet_Size", "Outlet_Location_Type", "Outlet_Type", "Item_Outlet_Sales",
]
raw = pd.read_csv(
    csv_path,
    dtype={"Item_Identifier": "string", "Outlet_Identifier": "string"},
)
if raw.columns.tolist() != expected_columns:
    raise ValueError("Schema CSV không khớp 12 cột BigMart mong đợi.")
if raw.shape != (8523, 12):
    raise ValueError(f"Kích thước CSV không mong đợi: {raw.shape}.")
if raw["Item_Outlet_Sales"].isna().any() or raw["Item_Outlet_Sales"].le(0).any():
    raise ValueError("Target có giá trị thiếu hoặc không dương.")

print(f"Số dòng, số cột: {raw.shape}")
print(f"Sản phẩm: {raw['Item_Identifier'].nunique()}; cửa hàng: {raw['Outlet_Identifier'].nunique()}")
print(f"Dòng trùng toàn bộ: {raw.duplicated().sum()}")
raw.head()

Số dòng, số cột: (8523, 12)
Sản phẩm: 1559; cửa hàng: 10
Dòng trùng toàn bộ: 0


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


## Quy trình tiếp theo

1. Chạy toàn bộ notebook này để xác minh nguồn, hash và schema.
2. Chạy `02_EDA.ipynb` để tạo dữ liệu xử lý, split và các đặc trưng train-only.
3. Chạy hai notebook mô hình trong thư mục `notebooks/` theo hướng dẫn tại README.